In [1]:
from typing import Dict, List, Callable, Any

## 工具类

In [2]:
class Tool:
    """
    工具属性说明：每个工具包括五部分
    name：唯一工具识别编号
    description：工具用途，便于LLM调用
    input_schema：工具期待的数据格式
    output_shcema：工具返回的格式、
    func：具体工作的函数
    """
    def __init__(
            self,
            name: str,
            description: str,
            input_schema: Dict[str, Any],
            output_schema: Dict[str, Any],
            func: Callable[..., Any],
    ):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.output_schema = output_schema
        self.func = func

    def __call__(self, **kwargs):
        return self.func(**kwargs)

## 工具注册
工具注册类用来记录工具及其用法，并随时进行管理和调用

In [3]:
from typing import Union, Literal
from pydantic import BaseModel

class ToolRegistry:
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
    
    def register(self, tool: Tool):
        self.tools[tool.name] = tool
    
    def get(self, name: str) -> Tool:
        if name not in self.tools.keys():
            raise ValueError(f"Tool '{name}' not found")
        return self.tools[name]
    
    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema.model_json_schema()
            } 
            for tool in self.tools.values()
        ]
    
    def get_tool_call_args_type(self) -> Union[BaseModel]:
        input_args_models = [tool.input_schema for tool in self.tools.values()]
        tool_call_args = Union[tuple(input_args_models)]
        return tool_call_args
    
    def get_tool_names(self) -> Literal[None]:
        return Literal[*self.tools.keys()]

## list_tools()函数告诉LLM它可以做什么
它返回如下格式的描述

In [4]:
[
    {
        "name": "add",
        "description": "Add two Number",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {"type": "integer"},
                "b": {"type": "integer"}
            },
            "required": ["a", "b"]
        }
    }
]

[{'name': 'add',
  'description': 'Add two Number',
  'input_schema': {'type': 'object',
   'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}},
   'required': ['a', 'b']}}]

## get_tool_names()
可以阻止大模型幻觉，其将所有的工具列表返回，强制让大模型选择

## 注册工具

In [5]:
def add(a: int, b: int) -> int:
    return a + b

def multiply(a: int, b: int) -> int:
    return a * b

In [6]:
class ToolAddArgs(BaseModel):
    a: int
    b: int

class ToolMultiplyArgs(BaseModel):
    a: int
    b: int

In [7]:
registry = ToolRegistry()

add_args = {
    "name": "add",
    "description": "Add two numbers",
    "input_schema": ToolAddArgs,
    "output_schema": {"result": "int"},
    "func": add
}

mul_args = {
    "name": "mul",
    "description": "Multiply two numbers",
    "input_schema": ToolMultiplyArgs,
    "output_schema": {"result": "int"},
    "func": multiply
}

registry.register(
    Tool(**add_args)
)

registry.register(
    Tool(**mul_args)
)

In [8]:
registry.get_tool_names()

typing.Literal['add', 'mul']

In [9]:
registry.list_tools()

[{'name': 'add',
  'description': 'Add two numbers',
  'input_schema': {'properties': {'a': {'title': 'A', 'type': 'integer'},
    'b': {'title': 'B', 'type': 'integer'}},
   'required': ['a', 'b'],
   'title': 'ToolAddArgs',
   'type': 'object'}},
 {'name': 'mul',
  'description': 'Multiply two numbers',
  'input_schema': {'properties': {'a': {'title': 'A', 'type': 'integer'},
    'b': {'title': 'B', 'type': 'integer'}},
   'required': ['a', 'b'],
   'title': 'ToolMultiplyArgs',
   'type': 'object'}}]

In [10]:
registry.get('add')

## Pydantic类型安全
使用Pydantic而不是JSON是因为类型安全考虑。当使用LLM时，最大的挑战是保证返回的数据格式可以被代码信任执行。即使今天大模型已经被广泛训练如何使用工具，但幻觉问题依然存在，这就是为什么我们需要保证类型安全。
Pydantic模型像合约一样运行：
1.自动验证输入数据
2.数据不合法时提供纠错信息
3.允许IDE自动补全
4.生成现代LLM可以使用的JSON schemas数据格式

In [11]:
ToolNameLiteral = registry.get_tool_names()
ToolArgsUnion = registry.get_tool_call_args_type()

class ToolCall(BaseModel):
    action: Literal["tool"]
    thought: str
    tool_name: ToolNameLiteral
    args: ToolArgsUnion

class FinalAnswer(BaseModel):
    action: Literal["final"]
    answer: str
LLMResponse = Union[ToolCall, FinalAnswer]

In [23]:
ToolArgsUnion

typing.Union[__main__.ToolAddArgs, __main__.ToolMultiplyArgs]

ReAct模式下，LLM必须选择一种行动类型（tool或者final）

https://pub.towardsai.net/creating-an-advanced-ai-agent-from-scratch-with-python-in-2025-part-1-ce74a23f6514

## LLM Wrapper
现在可以整合谷歌Gemini API。这里可以自行替换不同的LLM厂商，付费免费皆可。如果后续私有部署在集群上，可选择将模型参数放上去。

In [12]:
import json
from google import genai
from google.genai import types

class DeepSeekLLM:
    def __init__(self, client, tool_registry, model="gemini-2.5-flash"):
        self.client = client
        self.model = model
        self.tool_registry = tool_registry
        self.system_instruction = self._create_system_instruction()

    def _create_system_instruction(self) -> str:
        tools_description = json.dumps(
            self.tool_registry.list_tools(),
            indent=2
        )

        system_prompt = """
        你是一个对话AI agent，你可以使用外部工具。
        硬性规则（必须遵守的规则）：
        - 严禁在内部执行任何可由工具完成的操作；
        - 如果存在可执行任务任何部分的工具，必须使用该工具；
        - 严禁跳过工具，即使是简单或显而易见的步骤；
        - 严禁将多个操作合并为单个步骤，除非某个工具明确支持这样做；
        - 当且仅当没有任何工具能进一步推进任务时，你才可以生成最终答案。
        工具使用规则：
        - 每次工具调用必须执行且仅执行一个有意义的操作；
        - 如果任务需要多个操作，你必须按照顺序调用工具；
        - 如果多个工具都适用，选择最具体的哪一个。
        响应格式：
        - 你必须仅以有效的JSON格式进行响应；
        - 严禁在JSON之外包含任何解释说明；
        - 每次响应必须且仅能选择一个动作。
        工具调用格式：
        {
            "action": "tool",
            "thought": "...",
            "tool_name": "...",
            "inputs": { ... }
        }
        最终输出格式:
        {
            "action": "final",
            "answer": "..."
        }""" + "\\n\\nAvailable tools with description:\\n" + tools_description
        return system_prompt
    
    def _format_gemini_chat_history(self, history: list[dict]) -> list:
        formatted_history = []
        for message in history:
            if message["role"] == "user":
                formatted_history.append(types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                ))
            if message["role"] == "assistant":
                formatted_history.append(types.Content(
                    role="model",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                ))
            if message["role"] == "tool":
                formatted_history.append(types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=message["tool_name"],
                            response={"result": message["tool_response"]}
                        )
                    ]
                ))
        return formatted_history

    def generate(self, history: list[dict]) -> str:
        gemini_history_format = self._format_gemini_chat_history(history)
        response = self.client.models.generate_content(
            model=self.model,
            contents=gemini_history_format,
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type="application/json",
                response_schema=LLMResponse,
                system_instruction=self.system_instruction,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
            )
        )
        return response.text


## 为什么设置严格规则
LLMs总是试图帮助我们，会内部做数学运算或者推理。但我们需要我们的的代理行为是可见并且可信赖的。通过强迫它使用工具，我们可做到：
- 记录并debug每一个步骤
- 可切换工具而不是切换代理
- 可独立测试工具
- 可维持清晰的行为路径审计记录

In [13]:
def _format_gemini_chat_history(self, history: list[dict]) -> list:
    formatted_history = []
    for message in history:
        if message["role"] == "user":
            formatted_history.append(types.Content(
                role="user",
                parts=[
                    types.Part.from_text(text=message["content"])
                ]
            ))
        if message["role"] == "assistant":
            formatted_history.append(types.Content(
                role="model",
                parts=[
                    types.Part.from_text(text=message["content"])
                ]
            ))
        if message["role"] == "tool":
            formatted_history.append(types.Content(
                role="tool",
                parts=[
                    types.Part.from_function_response(
                        name=message["tool_name"],
                        response={"result": message["tool_response"]}
                    )
                ]
            ))
    return formatted_history

def generate(self, history: list[dict]) -> str:
    gemini_history_format = self._format_gemini_chat_history(history)
    response = self.client.models.generate_content(
        model=self.model,
        contents=gemini_history_format,
        config=types.GenerateContentConfig(
            temperature=0,
            response_mime_type="application/json",
            response_schema=LLMResponse,
            system_instruction=self.system_instruction,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
        )
    )
    return response.text

## 参数
- temperature: 我们希望确定的、一致性的行为
- response_mime_type: 强制输出JSON格式
- response_schema:使用Pydantic数据验证模型
- automatic_function_calling_disabled: 手动控制工具执行

In [30]:
class Agent:
    def __init__(self, llm, tool_registry, max_steps=5):
        self.llm = llm
        self.tool_registry = tool_registry
        self.history = []
        self.max_steps = max_steps

    def run(self, user_input: str):
        self.history.append({"role": "user", "content": user_input})
        for step in range(self.max_steps):
            # 获取LLM决策
            llm_output = self.llm.generate(self.history)
            print(llm_output)
            action = json.loads(llm_output)
            print(f"这是思考过程 {self.history}")
            if action["action"] == "tool":
                # 记录思考过程
                self.history.append(
                    {"role": "assistance", "content": llm_output}
                )
                # 执行工具
                tool = self.tool_registry.get(action["tool_name"])
                result = tool(**action["args"])
                # 记录结果
                observation = f"Tool {tool.name} retruned: {result}"
                self.history.append(
                    {"role": "tool", "tool_name": tool.name, "tool_response": result}
                )
                continue
            if action["action"] == "final":
                self.history.append(
                    {"role": "assistant", "content": llm_output}
                )
                return action["answer"]
        raise RuntimeError("Agent 没有在最大步数前完成")

In [32]:
from google import genai

# 初始化代理
client = genai.Client(api_key='')
# 创建LLM及代理
llm = DeepSeekLLM(client, registry)
agent = Agent(llm, registry)
def chat_with_agent(agent: Agent):
    print("欢迎，输入'exit'退出\\n")
    while True:
        user_input = input("User:")
        if user_input.lower() in ["exit", "quit", "q"]:
            print("再见！")
            break
        try:
            response = agent.run(user_input)
            print(f"Agent: {response}")
        except RuntimeError as e:
            print(f"Agent 错误： {e}")
        except Exception as e:
            print(f"Unexpected error: {e}")

# 开始对话
chat_with_agent(agent)

欢迎，输入'exit'退出\n
{
  "action": "tool",
  "thought": "The user wants to calculate (5+2+3+5+1) * 10. I need to perform the additions first. I will start by adding 5 and 2.",
  "tool_name": "add",
  "args": {
    "a": 5,
    "b": 2
  }
}
这是思考过程 [{'role': 'user', 'content': '5+2+3+5+1的结果乘10最后是多少'}]
{
  "action": "tool",
  "thought": "I need to continue adding the numbers. The previous sum was 7 (from 5+2). Now I need to add 3 to 7.",
  "tool_name": "add",
  "args": {
    "a": 7,
    "b": 3
  }
}
这是思考过程 [{'role': 'user', 'content': '5+2+3+5+1的结果乘10最后是多少'}, {'role': 'assistance', 'content': '{\n  "action": "tool",\n  "thought": "The user wants to calculate (5+2+3+5+1) * 10. I need to perform the additions first. I will start by adding 5 and 2.",\n  "tool_name": "add",\n  "args": {\n    "a": 5,\n    "b": 2\n  }\n}'}, {'role': 'tool', 'tool_name': 'add', 'tool_response': 7}]
{
  "action": "tool",
  "thought": "I need to continue adding the numbers. The current sum is 10. I need to add the next 

In [26]:
agent.history

[{'role': 'user', 'content': '5+2结果乘10最后是多少'},
 {'role': 'user', 'content': '5+2结果乘10，最后得多少'}]

history可以有效进行记录，便于Agent debug

## 为什么这个架构很重要
- 1.可扩展性。可自行添加工具并注册
- 2.供应商灵活性，自由切换LLM服务商
- 3.可测试。可独立测试工具，验证端到端工作流
- 4.可观测性。每一步都记录在历史之中

## 接下来，继续拓展
- 1.长时记忆。使用vector数据库进行对话记录，并从对话中学习
- 2.Human in the Loop（HITL）。对风险行为暂停获取人工许可
- 3.提高可观测性。日志、追踪以及管理
- 4.错误恢复。处理工具错误并且尝试重试逻辑
https://pub.towardsai.net/creating-an-advanced-ai-agent-from-scratch-with-python-in-2026-part-2-0f41c8d80bff

## Part1局限性
- 无记忆功能
- 无检查功能：agent可以不经过人类同意执行动作
- 有限的可视性：无法有效debug
- 碎片化执行：工具失败会冲击整个agent
## 我们将添加
- 长期记忆：通过会话保存对话
- HITL：部分动作要求人类许可
- 卓越的可监控性：完全的日志和追踪
- 错误恢复：重试逻辑

### 特性1：长期记忆
### 记忆vs上下文窗口
不存储所有的信息是因为上下文窗口限制，超长对话很容易超过上下文限制，会造成：
- 截断的历史
- 高延迟
- 更高的消费

解决办法是选择性记忆：选择保存重要信息，对每个请求只注入相关的上下文信息

### 设计记忆系统
记忆系统需要包含四种能力：
- 一致性：会话重启
- 会话感知：追踪那些对话相似
- 选择性回溯：只载入最近或者相关的记忆
- 清理隔离：记忆不与当前的对话交互

In [2]:
class MemoryStore:
    def __init__(self, file_path: str, max_entries: int = 50):
        self.file_path = file_path
        self.max_entries = max_entries
        self._ensure_file()

    def _ensure_file(self):
        if not os.path.exists(self.file_path):
            with open(self.file_path, 'w') as f:
                json.dump([], f)

    def load_all(self) -> List[dict]:
        try:
            with open(self.file_path, 'r') as f:
                return json.laod(f)
        except Exception:
            return []
        
    def append(self, entry: dict):
        data = self.load_all()
        data.append(entry)

        with open(self.file_path, 'w') as f:
            json.dump(data, f, indent=2)

    def get_recent(self, limit: Optional[int] = None) -> list[dict]:
        data = self.load_all()
        limit = limit or self.max_entries
        return data[-limit:]
    
    def delete_all(self):
        with open(self.file_path, 'w') as f:
            json.dump([], f)



NameError: name 'List' is not defined

In [ ]:
class Agent:
    def __init__(self, llm, tool_registry, memory_store, max_steps=5, memory_injection_limit=6, ):
        self.llm = llm
        self.tool_registry = tool_registry
        self.history = []
        self.max_steps = max_steps
        self.memory_injection_limit = memory_injection_limit
        self.memory_store = memory_store

    def run(self, user_input: str):
        self.history.append({"role": "user", "content": user_input})

        for step in range(self.max_steps):
            # 获取LLM决策
            llm_output = self.llm.generate(self.history)
            print(llm_output)
            action = json.loads(llm_output)
            print(f"这是思考过程 {self.history}")
            if action["action"] == "tool":
                # 记录思考过程
                self.history.append(
                    {"role": "assistance", "content": llm_output}
                )
                # 执行工具
                tool = self.tool_registry.get(action["tool_name"])
                result = tool(**action["args"])
                # 记录结果
                observation = f"Tool {tool.name} retruned: {result}"
                self.history.append(
                    {"role": "tool", "tool_name": tool.name, "tool_response": result}
                )
                continue
            if action["action"] == "final":
                self.history.append(
                    {"role": "assistant", "content": llm_output}
                )
                timestamp = datetime.now(UTC).isoformat()

                self.memory_store.append({
                    "session_id" :self.session_id,
                    "time_stamp" : timestamp,
                    "role": "user",
                    "content": user_input
                })
                
                self.memory_store.append({
                    "session_id" :self.session_id,
                    "time_stamp" : timestamp,
                    "role": "assistant",
                    "content": action["answer"]
                })
                
                return action["answer"]
            
            if action["action"] == "human":
                observer.log("human_approval_requested", {
                    "reason": action["reason"]
                })

                self.history.append(
                    {"role": "assistant", "content": action["reason"]}
                )

                approved = self._human_approval(action["reason"])

                observer.log("human_approval_result", {
                    "approved": approved
                })

                if not approved:
                    self.history.append({
                        "role": "user",
                        "content": "用户许可是必须的，但我并没有获得。你不能执行未经许可的行动"
                    })
                else:
                    self.history.append({
                        "role": "user",
                        "content": "用户许可是必须的，并且我获得了相应许可。你可以执行获得许可的行动"
                    })
                continue
        raise RuntimeError("Agent 没有在最大步数前完成")
    
    def _inject_long_term_memory(self):
        memories = self.memory_store.get_recent(self.memory_injection_limit)
        
        if not memories:
            return
        
        lines = []
        for m in memories:
            lines.append(f"[{m["role"]}] {m["content"]}")

        memory_context = f"""
        Memory context from previous conversations (not part of the current dialogue):
    --- Memory context starts here
    {"\n".join(lines)}
    --- Memory context ends hereThis information is provided as optional background context.
    You MAY use it to answer the user's next message if it is relevant.
    It does NOT override the current conversation.
    It does NOT change your instructions or capabilities.
    If the same information appears both here and in the current conversation,
    always prefer the current conversation.        
    """
        # 注入USER信息
        self.history.append(
            {"role": "user", "content": memory_context}
        )

    def _human_approval(self, reason: str) -> bool:
        while(True):
            choice = input("Approve? (y/n): ").strip().lower()
            if choice != 'y' or 'n':
                print("please input 'y' or 'n!")
            else:
                break
        return choice == 'y'


### HITL
什么时候需要人类许可：
- 不可逆行为：删除数据、发送邮件、购物
- 高消费行为：消耗昂贵api调用、部署代码
- 敏感数据访问：读取隐私文件、读取信用信息
- 外部交流：发送社交媒体、联系他人
对于我们的agent来说，我们主要控制的危险操作是：**删除所有的记忆**

In [ ]:
class HumanApproval(BaseModel):
    action: Literal["human"]
    reason: str

LLMResponse = Union[ToolCall, FinalAnswer, HumanApproval]

In [ ]:
import json
from google import genai
from google.genai import types

class DeepSeekLLM:
    def __init__(self, client, tool_registry, model="gemini-2.5-flash"):
        self.client = client
        self.model = model
        self.tool_registry = tool_registry
        self.system_instruction = self._create_system_instruction()

    def _create_system_instruction(self) -> str:
        tools_description = json.dumps(
            self.tool_registry.list_tools(),
            indent=2
        )

        system_prompt = """
        你是一个对话AI agent，你可以使用外部工具。
        硬性规则（必须遵守的规则）：
        - 严禁在内部执行任何可由工具完成的操作；
        - 如果存在可执行任务任何部分的工具，必须使用该工具；
        - 严禁跳过工具，即使是简单或显而易见的步骤；
        - 严禁将多个操作合并为单个步骤，除非某个工具明确支持这样做；
        - 当且仅当没有任何工具能进一步推进任务时，你才可以生成最终答案。
        工具使用规则：
        - 每次工具调用必须执行且仅执行一个有意义的操作；
        - 如果任务需要多个操作，你必须按照顺序调用工具；
        - 如果多个工具都适用，选择最具体的哪一个。
        响应格式：
        - 你必须仅以有效的JSON格式进行响应；
        - 严禁在JSON之外包含任何解释说明；
        - 每次响应必须且仅能选择一个动作。
        人类干预循环（）：
        - 你有一个特殊的行为叫做"human";
        - 你在操作任何不可逆、有破坏性以及敏感操作时必须选择"human"的行为
        - 包括但不限于：删除记忆，重置状态以及永远警惕存储数据
        - 当使用"human"行为时，你必须清楚的解释同意的理由
        - 在询问人类许可之后哦，你必须基于回应做出以下操作：
         1.如果获得许可：你必须继续那个任务，通过合适的操作（通常是一个工具调用）
         2.如果禁止执行：你必须告知用户原始的操作不会执行因为未获得许可
        - 不要重复连续的行动。你必须永远在一个"human"操作之后跟随"tool"调用
        工具调用格式：
        {
            "action": "tool",
            "thought": "...",
            "tool_name": "...",
            "inputs": { ... }
        }
        最终输出格式:
        {
            "action": "final",
            "answer": "..."
        }""" + "\\n\\nAvailable tools with description:\\n" + tools_description
        return system_prompt
    
    def _format_gemini_chat_history(self, history: list[dict]) -> list:
        formatted_history = []
        for message in history:
            if message["role"] == "user":
                formatted_history.append(types.Content(
                    role="user",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                ))
            if message["role"] == "assistant":
                formatted_history.append(types.Content(
                    role="model",
                    parts=[
                        types.Part.from_text(text=message["content"])
                    ]
                ))
            if message["role"] == "tool":
                formatted_history.append(types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=message["tool_name"],
                            response={"result": message["tool_response"]}
                        )
                    ]
                ))
        return formatted_history

    def generate(self, history: list[dict]) -> str:
        gemini_history_format = self._format_gemini_chat_history(history)
        response = self.client.models.generate_content(
            model=self.model,
            contents=gemini_history_format,
            config=types.GenerateContentConfig(
                temperature=0,
                response_mime_type="application/json",
                response_schema=LLMResponse,
                system_instruction=self.system_instruction,
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
            )
        )
        return response.text